In [ ]:
import copy
import os
import random
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr

from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [ ]:
RNG_SEED = 42

def seed_all(seed: int = RNG_SEED) -> None:
    """
    Set random seeds for reproducibility across Python, NumPy,
    and hash-based operations.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


In [ ]:
import pandas as pd
import requests

article_id = 31395765
file_id = 67453365

files_url = f"https://api.figshare.com/v2/articles/{article_id}/files"
files = requests.get(files_url).json()

file_info = next(f for f in files if f["id"] == file_id)

Data001 = pd.read_csv(file_info["download_url"])

print("Loaded:", file_info["name"])
print("Shape:", Data001.shape)

display(Data001.head())

Loaded: Data001_ReadyForCoxPH_v2(2026-08-12).csv
Shape: (50000, 12)


,IRSD_quintile,Age,smoking_status,BMI,diabetes,CKD,HbA1c,eGFR,SBP,AF,cvd_event,cvd_time
0,4,50.395265,non,28.166346,0,0,4.317890,83.077560,118.668194,0,1,2.963423
1,3,39.226761,ex,16.825992,0,0,4.700951,81.488401,123.719732,0,0,4.407252
2,5,55.489004,non,23.523419,0,0,3.669685,86.779230,126.650517,0,0,4.490739
3,4,51.910529,non,31.981932,0,0,4.486977,94.704093,113.889670,0,0,4.888965
4,1,47.091570,ex,25.351159,0,0,4.315440,86.336256,125.912615,0,0,4.490326


In [ ]:
seed_all()
###===######===######===######===######===######===######===######===###
###===######===######===######===######===######===######===######===###
###===######===######===######===######===######===######===######===###
df1 = copy.copy(Data001)

df_emr = pd.DataFrame([])

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
X = df1.index.astype(int)
df_emr['Patient_ID'] = (X**2 - 77) * 3 + 500

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
df_emr['Age_At_2024'] = np.round(df1['Age'] + 7, 2)

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
df_emr['SMOKING_STATUS'] = df1['smoking_status']

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
mask_non = df_emr['SMOKING_STATUS'].eq('non')

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
idx_to_nan = df_emr[mask_non].sample(frac=0.1566, random_state=42).index
df_emr.loc[idx_to_nan, 'SMOKING_STATUS'] = np.nan

df_emr['IRSD_Quintile'] = df1['IRSD_quintile']

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
df_emr['CVD_Event'] = df1['cvd_event'].astype(int)

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
base_date = pd.to_datetime('2017-01-01')

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
event_dates = base_date + pd.to_timedelta(df1['cvd_time'] * 365.25, unit='D')
df_emr['CVD_Time'] = event_dates.dt.strftime('%Y-%m')

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
df_emr.loc[df_emr['CVD_Event'] == 0, 'CVD_Time'] = '2022-12'


In [ ]:
df_emr

,Patient_ID,Age_At_2024,SMOKING_STATUS,IRSD_Quintile,CVD_Event,CVD_Time
0,269,57.40,non,4,1,2019-12
1,272,46.23,ex,3,0,2022-12
2,281,62.49,non,5,0,2022-12
3,296,58.91,non,4,0,2022-12
4,317,54.09,ex,1,0,2022-12
...,...,...,...,...,...,...
49995,7498500344,78.01,ex,5,0,2022-12
49996,7498800317,52.88,current,3,0,2022-12
49997,7499100296,70.10,non,5,0,2022-12
49998,7499400281,44.62,non,3,0,2022-12


In [ ]:
seed_all()
###===######===######===######===######===######===######===######===###
###===######===######===######===######===######===######===######===###
###===######===######===######===######===######===######===######===###
X = df1.index.to_series().astype(int)
patient_id = (X**2 - 77) * 3 + 500

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
flags = df1[['diabetes', 'CKD', 'AF']].astype(int).copy()
flags['Patient_ID'] = patient_id.values

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
mask_dm = flags['diabetes'] == 1
df_dm = pd.DataFrame({
    'Patient_ID': flags.loc[mask_dm, 'Patient_ID'],
    'Category': 'Diabetes'
})

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
mask_ckd = flags['CKD'] == 1
df_ckd = pd.DataFrame({
    'Patient_ID': flags.loc[mask_ckd, 'Patient_ID'],
    'Category': 'Chronic Kidney Disease'
})

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
mask_af = flags['AF'] == 1
df_af = pd.DataFrame({
    'Patient_ID': flags.loc[mask_af, 'Patient_ID'],
    'Category': 'Atrial Fibrillation'
})

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
df_flags_long = pd.concat([df_dm, df_ckd, df_af], ignore_index=True)

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
rng_dates = np.random.default_rng(42)

months = pd.date_range('2012-01-01', '2016-12-01', freq='MS')
n_rows = len(df_flags_long)

random_months = rng_dates.choice(months, size=n_rows, replace=True)
df_flags_long['Date'] = pd.to_datetime(random_months).strftime('%Y-%m')

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
rng_terms = np.random.default_rng(123)

mask_dm_cat = df_flags_long['Category'] == 'Diabetes'
choices_dm = np.array([
    "Diabetes",
    "T2DM",
    "ICD10: E11",
    "ICD9:250",
    "High blood sugar",
    "High glucose",
])
probs_dm = np.array([0.40, 0.20, 0.15, 0.15, 0.05, 0.05])

df_flags_long.loc[mask_dm_cat, 'Category'] = rng_terms.choice(
    choices_dm,
    size=mask_dm_cat.sum(),
    p=probs_dm
)

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
mask_af_cat = df_flags_long['Category'] == 'Atrial Fibrillation'
choices_af = np.array([
    "AF",
    "Atrial fibrillation",
    "AFib",
    "A-fib",
    "ICD10: I48",
    "ICD9: 427.31",
])
probs_af = np.array([0.45, 0.30, 0.075, 0.075, 0.05, 0.05])

df_flags_long.loc[mask_af_cat, 'Category'] = rng_terms.choice(
    choices_af,
    size=mask_af_cat.sum(),
    p=probs_af
)

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
mask_ckd_cat = df_flags_long['Category'] == 'Chronic Kidney Disease'
choices_ckd = np.array([
    "CKD",
    "Chronic kidney disease",
    "Chronic renal failure",
    "CRF",
    "Renal insufficiency",
    "ICD10: N18",
    "ICD9: 585",
])
probs_ckd = np.array([0.45, 0.25, 0.05, 0.05, 0.05, 0.10, 0.05])

df_flags_long.loc[mask_ckd_cat, 'Category'] = rng_terms.choice(
    choices_ckd,
    size=mask_ckd_cat.sum(),
    p=probs_ckd
)

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
df_flags = df_flags_long.sample(frac=1.0, random_state=999).reset_index(drop=True)


In [ ]:
seed_all()
###===######===######===######===######===######===######===######===###
###===######===######===######===######===######===######===######===###
###===######===######===######===######===######===######===######===###
X = df1.index.to_series().astype(int)
patient_id = (X**2 - 77) * 3 + 500

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
df_hba1c = pd.DataFrame({
    "Patient_ID": patient_id.values,
    "Measure": "HbA1c",
    "Value": df1["HbA1c"].values,
})

df_egfr = pd.DataFrame({
    "Patient_ID": patient_id.values,
    "Measure": "eGFR",
    "Value": df1["eGFR"].values,
})

df_sbp = pd.DataFrame({
    "Patient_ID": patient_id.values,
    "Measure": "SBP",
    "Value": df1["SBP"].values,
})

baseline_biomarkers = pd.concat([df_hba1c, df_egfr, df_sbp], ignore_index=True)

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
baseline_biomarkers["Description"] = baseline_biomarkers["Measure"]

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
rng_dates = np.random.default_rng(42)

months = pd.date_range("2012-01-01", "2016-12-01", freq="MS")
n_rows = len(baseline_biomarkers)

random_months = rng_dates.choice(months, size=n_rows, replace=True)
baseline_biomarkers["Date"] = pd.to_datetime(random_months).strftime("%Y-%m")

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
rng_terms = np.random.default_rng(123)

mask_hba1c = baseline_biomarkers["Measure"] == "HbA1c"

choices_hba1c = np.array([
    "HbA1c",                    # 55%
    "HBA1C",                    # 15%
    "A1C",                      # 10%
    "Hb A1c",                   # 5%
    "Glycated hemoglobin",      # 2.5%
    "Glycosylated hemoglobin",  # 2.5%
    "HA1C",                     # 1.5%
    "HbAlc",                    # 1.5%
    "LOINC: 4548-4",            # 4%
    "HbA1c mmol/mol",           # 3%
])

probs_hba1c = np.array([
    0.55,  # HbA1c
    0.15,  # HBA1C
    0.10,  # A1C
    0.05,  # Hb A1c
    0.025, # Glycated hemoglobin
    0.025, # Glycosylated hemoglobin
    0.015, # HA1C
    0.015, # HbAlc
    0.04,  # LOINC: 4548-4
    0.03,  # HbA1c mmol/mol
])
probs_hba1c = probs_hba1c / probs_hba1c.sum()

baseline_biomarkers.loc[mask_hba1c, "Description"] = rng_terms.choice(
    choices_hba1c,
    size=mask_hba1c.sum(),
    p=probs_hba1c
)

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
mask_egfr = baseline_biomarkers["Measure"] == "eGFR"

choices_egfr = np.array([
    "eGFR",                        # 60%
    "EGFR",                        # 20%
    "eGFR (mL/min/1.73m²)",        # 8%
    "GFR",                         # 5%
    "Estimated GFR",               # 3%
    "GFR-e",                       # 1%
    "e-GFR",                       # 1%
    "e GFR",                       # ~0.33%
    "EGfr",                        # ~0.33%
    "eGFr",                        # ~0.33%
])

probs_egfr = np.array([
    0.60,   # eGFR
    0.20,   # EGFR
    0.08,   # eGFR (mL/min/1.73m²)
    0.05,   # GFR
    0.03,   # Estimated GFR
    0.01,   # GFR-e
    0.01,   # e-GFR
    0.0033, # e GFR
    0.0033, # EGfr
    0.0034, # eGFr
])
probs_egfr = probs_egfr / probs_egfr.sum()

baseline_biomarkers.loc[mask_egfr, "Description"] = rng_terms.choice(
    choices_egfr,
    size=mask_egfr.sum(),
    p=probs_egfr
)

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
mask_sbp = baseline_biomarkers["Measure"] == "SBP"

choices_sbp = np.array([
    "SBP",                      # 50%
    "Systolic BP",              # 20%
    "Systolic Blood Pressure",  # 10%
    "BP Systolic",              # 8%
    "BP_sys",                   # ~1.67%
    "BP SYS",                   # ~1.67%
    "BP_Systolic",              # ~1.66%
    "Blood Pressure – Systolic",# 3%
    "SBP (mmHg)",               # 2%
    "SyBP",                     # ~0.67%
    "Syst BP",                  # ~0.67%
    "SystolicBP",               # ~0.66%
])

probs_sbp = np.array([
    0.50,    # SBP
    0.20,    # Systolic BP
    0.10,    # Systolic Blood Pressure
    0.08,    # BP Systolic
    0.0167,  # BP_sys
    0.0167,  # BP SYS
    0.0166,  # BP_Systolic
    0.03,    # Blood Pressure – Systolic
    0.02,    # SBP (mmHg)
    0.0067,  # SyBP
    0.0067,  # Syst BP
    0.0066,  # SystolicBP
])
probs_sbp = probs_sbp / probs_sbp.sum()

baseline_biomarkers.loc[mask_sbp, "Description"] = rng_terms.choice(
    choices_sbp,
    size=mask_sbp.sum(),
    p=probs_sbp
)

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
baseline_biomarkers["Unit"] = None

baseline_biomarkers.loc[mask_sbp, "Unit"] = "mmHg"
baseline_biomarkers.loc[mask_egfr, "Unit"] = "mL/min/1.73m²"
baseline_biomarkers.loc[mask_hba1c, "Unit"] = "%"

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
rng_conv = np.random.default_rng(2025)

mask_hba1c_all = mask_hba1c  # alias
n_hba1c = mask_hba1c_all.sum()
n_convert = int(np.floor(0.05 * n_hba1c))

hba1c_idx = baseline_biomarkers.index[mask_hba1c_all]
convert_idx = rng_conv.choice(hba1c_idx, size=n_convert, replace=False)

hba1c_percent_vals = baseline_biomarkers.loc[convert_idx, "Value"].astype(float)
hba1c_mmolmol = (hba1c_percent_vals - 2.15) * 10.929

baseline_biomarkers.loc[convert_idx, "Value"] = hba1c_mmolmol
baseline_biomarkers.loc[convert_idx, "Unit"] = "mmol/mol"

#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>#>>===>>>
baseline_biomarkers = (
    baseline_biomarkers
    .sample(frac=1.0, random_state=999)
    .reset_index(drop=True)
)


In [ ]:
baseline_biomarkers.head()

,Patient_ID,Measure,Value,Description,Date,Unit
0,5386159421,eGFR,77.238713,eGFR,2014-12,mL/min/1.73m²
1,5790852944,HbA1c,4.501069,HBA1C,2013-02,%
2,3141314912,HbA1c,4.161843,Hb A1c,2016-09,%
3,5573606096,eGFR,89.524620,eGFR,2013-10,mL/min/1.73m²
4,3311104121,SBP,131.546136,SBP,2013-11,mmHg


In [ ]:
seed_all()
rng = np.random.default_rng(2026)

###===######===######===######===######===######===######===######===###
###===######===######===######===######===######===######===######===###
###===######===######===######===######===######===######===######===###

n = len(df1)
ids = df1.index.to_series().astype(int)
patient_id_series = (ids**2 - 77) * 3 + 500

n_bmi_direct = int(np.floor(0.80 * n))
direct_idx = rng.choice(df1.index, size=n_bmi_direct, replace=False)

split_idx = df1.index.difference(direct_idx)

rows = []
for i in direct_idx:
    pid = patient_id_series.loc[i]
    bmi_val = df1.loc[i, "BMI"]
    rows.append({
        "Patient_ID": pid,
        "Measure": "BMI",
        "Value": round(float(bmi_val), 2),
        "Unit": np.nan,
        "Description": "BMI",
        "Date": pd.Timestamp(rng.choice(months)).strftime("%Y-%m")
    })

male_mean, male_sd = 175.6, 7.0
female_mean, female_sd = 161.8, 6.5

n_height_feet = int(np.floor(0.10 * len(split_idx)))
height_feet_idx = rng.choice(split_idx, size=n_height_feet, replace=False)

for i in split_idx:
    pid = patient_id_series.loc[i]
    u = rng.random()
    if u > 0.5:
        sex = "M"
        h_cm = rng.normal(loc=male_mean, scale=male_sd)
    else:
        sex = "F"
        h_cm = rng.normal(loc=female_mean, scale=female_sd)
    h_cm = float(np.clip(h_cm, 140.0, 210.0))
    height_m = h_cm / 100.0
    bmi_val = df1.loc[i, "BMI"]
    weight_kg = bmi_val * (height_m ** 2)
    h_cm_r = round(h_cm, 1)
    w_kg_r = round(weight_kg, 1)

    if i in height_feet_idx:
        total_inches = h_cm / 2.54
        feet = int(total_inches // 12)
        inches = int(round(total_inches - feet*12))
        feet_str = f"{feet}'{inches}\""
        rows.append({
            "Patient_ID": pid,
            "Measure": "Height",
            "Value": feet_str,
            "Unit": "feet-and-inch",
            "Description": "Height",
            "Date": pd.Timestamp(rng.choice(months)).strftime("%Y-%m")
        })
    else:
        rows.append({
            "Patient_ID": pid,
            "Measure": "Height",
            "Value": h_cm_r,
            "Unit": "cm",
            "Description": "Height",
            "Date": pd.Timestamp(rng.choice(months)).strftime("%Y-%m")
        })

    rows.append({
        "Patient_ID": pid,
        "Measure": "Weight",
        "Value": w_kg_r,
        "Unit": "kg",
        "Description": "Weight",
        "Date": pd.Timestamp(rng.choice(months)).strftime("%Y-%m")
    })

df_bmi_new = pd.DataFrame(rows)

In [ ]:
df_bmi_new.head()

,Patient_ID,Measure,Value,Unit,Description,Date
0,1554599357,BMI,31.97,NaN,BMI,2016-08
1,379687769,BMI,27.12,NaN,BMI,2014-06
2,6374523917,BMI,27.86,NaN,BMI,2016-02
3,434596157,BMI,24.17,NaN,BMI,2013-03
4,4950928397,BMI,22.92,NaN,BMI,2015-05


In [ ]:
seed_all()
###===######===######===######===######===######===######===######===###
###===######===######===######===######===######===######===######===###
###===######===######===######===######===######===######===######===###
baseline_biomarkers = pd.concat([baseline_biomarkers, df_bmi_new], ignore_index=True)
baseline_biomarkers = baseline_biomarkers.sample(frac=1.0, random_state=999).reset_index(drop=True)
baseline_biomarkers = baseline_biomarkers.drop(['Measure'], axis = 1)

In [ ]:
baseline_biomarkers

,Patient_ID,Value,Description,Date,Unit
0,819127997,78.087308,eGFR,2014-03,mL/min/1.73m²
1,3847713176,8.369466,HbA1c,2015-02,%
2,3508304696,140.328005,SBP,2016-09,mmHg
3,1685212472,69.99202,GFR,2015-06,mL/min/1.73m²
4,1105805072,29.95,BMI,2015-04,NaN
...,...,...,...,...,...
209995,455199641,85.166227,eGFR,2014-09,mL/min/1.73m²
209996,3314294201,3.988246,HbA1c,2012-08,%
209997,5555517536,5.87664,A1C,2015-11,%
209998,6964515641,156.4,Height,2012-03,cm


In [ ]:
df_emr

,Patient_ID,Age_At_2024,SMOKING_STATUS,IRSD_Quintile,CVD_Event,CVD_Time
0,269,57.40,non,4,1,2019-12
1,272,46.23,ex,3,0,2022-12
2,281,62.49,non,5,0,2022-12
3,296,58.91,non,4,0,2022-12
4,317,54.09,ex,1,0,2022-12
...,...,...,...,...,...,...
49995,7498500344,78.01,ex,5,0,2022-12
49996,7498800317,52.88,current,3,0,2022-12
49997,7499100296,70.10,non,5,0,2022-12
49998,7499400281,44.62,non,3,0,2022-12


In [ ]:
df_flags

,Patient_ID,Category,Date
0,1338163469,Diabetes,2012-05
1,150591944,Diabetes,2014-12
2,545940569,ICD10: E11,2016-10
3,7134075944,Diabetes,2013-02
4,5959722392,Diabetes,2014-09
...,...,...,...
4412,503030072,Diabetes,2012-09
4413,21547469,ICD10: E11,2012-06
4414,3405452861,ICD9:250,2014-01
4415,23335832,High glucose,2013-07


In [ ]:
baseline_biomarkers

,Patient_ID,Value,Description,Date,Unit
0,819127997,78.087308,eGFR,2014-03,mL/min/1.73m²
1,3847713176,8.369466,HbA1c,2015-02,%
2,3508304696,140.328005,SBP,2016-09,mmHg
3,1685212472,69.99202,GFR,2015-06,mL/min/1.73m²
4,1105805072,29.95,BMI,2015-04,NaN
...,...,...,...,...,...
209995,455199641,85.166227,eGFR,2014-09,mL/min/1.73m²
209996,3314294201,3.988246,HbA1c,2012-08,%
209997,5555517536,5.87664,A1C,2015-11,%
209998,6964515641,156.4,Height,2012-03,cm
